<a href="https://colab.research.google.com/github/muajnstu/Large_Scale_Implementation_of_DSK_Chain/blob/main/Downstram_Pipeline_of_Proposed_Method(HOTEL_DATA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, f1_score)
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, roc_auc_score, recall_score, precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier
)

from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    Perceptron,
    SGDClassifier,
    PassiveAggressiveClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

In [ ]:
df = df = pd.read_csv('https://raw.githubusercontent.com/muajnstu/Large_Scale_Implementation_of_DSK_Chain/refs/heads/main/filtered%20data/Saimon%20and%20Roni/Clustered%20Hotel%20Booking%20Data.csv')

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

In [ ]:
y_cat = pd.Categorical(y)
y_codes = y_cat.codes

**Applying Random over sampler(ros) on whole dataset**

In [ ]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
print("Class distribution after ros:", pd.Series(y_resampled).value_counts())

Class distribution after ros: is_canceled
4     9577
3     9577
0     9577
1     9577
8     9577
5     9577
6     9577
7     9577
2     9577
10    9577
11    9577
12    9577
9     9577
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=46, stratify=y_resampled
)

**Implementation**

In [ ]:
def print_metrics(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    num_classes = cm.shape[0]

    if num_classes == 2:
        TN, FP, FN, TP = cm.ravel()
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        gmean = np.sqrt(specificity * sensitivity)
        type1 = FP / (FP + TN) if (FP + TN) > 0 else 0
        type2 = FN / (TP + FN) if (TP + FN) > 0 else 0
        fmeasure = f1_score(y_true, y_pred, pos_label=1)
        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob[:, 1])
            except Exception:
                auc = 0
    else:
        TP = np.diag(cm)
        FP = np.sum(cm, axis=0) - TP
        FN = np.sum(cm, axis=1) - TP
        TN = np.sum(cm) - (FP + FN + TP)

        specificity = np.mean([
            TN[i] / (TN[i] + FP[i]) if (TN[i] + FP[i]) > 0 else 0 for i in range(num_classes)
        ])
        sensitivity = np.mean([
            TP[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        gmean = np.sqrt(specificity * sensitivity)
        type1 = np.mean([
            FP[i] / (FP[i] + TN[i]) if (FP[i] + TN[i]) > 0 else 0 for i in range(num_classes)
        ])
        type2 = np.mean([
            FN[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        fmeasure = f1_score(y_true, y_pred, average='macro')
        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
            except Exception:
                auc = 0

    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Sensitivity   : {sensitivity:.4f}")
    print(f"Specificity   : {specificity:.4f}")
    print(f"G-Mean        : {gmean:.4f}")
    print(f"Type I Error  : {type1:.4f}")
    print(f"Type II Error : {type2:.4f}")
    print(f"F1 Score      : {fmeasure:.4f}")
    print(f"AUROC         : {auc:.4f}")

In [ ]:
def run_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
    except Exception:
        y_prob = None
    print(f"\nModel: {name}")
    print_metrics(y_test, y_pred, y_prob)

In [ ]:
ml_models = {
    # Original models
    "RandomForest": RandomForestClassifier(random_state=42),
    "ExtraTrees": ExtraTreesClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=15000, random_state=42),
    "RidgeClassifier": RidgeClassifier(random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "Perceptron": Perceptron(random_state=42),
    "SGDClassifier": SGDClassifier(random_state=42),
    #"KNN": KNeighborsClassifier(n_neighbors=3),
    "PassiveAggressive": PassiveAggressiveClassifier(random_state=42),
    #"RBFSVM": SVC(kernel='rbf', probability=True, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.01),
    "LightGBM": LGBMClassifier(verbosity=-1, random_state=42),


    "VotingSoft": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='soft',
        n_jobs=-1
    ),


    "VotingHard": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='hard',
        n_jobs=-1
    ),


    "VotingWeighted": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        voting='soft',
        weights=[2, 3, 2, 3, 3],
        n_jobs=-1
    )


}

In [ ]:
ml_models_part1 = {}
ml_models_part2 = {}

# Split ml_models into two parts
model_names = list(ml_models.keys())

for i, name in enumerate(model_names):
    if i < 10:
        ml_models_part1[name] = ml_models[name]
    else:
        ml_models_part2[name] = ml_models[name]

print(f"ml_models_part1 contains {len(ml_models_part1)} models: {list(ml_models_part1.keys())}")
print(f"ml_models_part2 contains {len(ml_models_part2)} models: {list(ml_models_part2.keys())}")

ml_models_part1 contains 10 models: ['RandomForest', 'ExtraTrees', 'Bagging', 'GradientBoosting', 'LogisticRegression', 'RidgeClassifier', 'DecisionTree', 'NaiveBayes', 'Perceptron', 'SGDClassifier']
ml_models_part2 contains 7 models: ['PassiveAggressive', 'LDA', 'QDA', 'LightGBM', 'VotingSoft', 'VotingHard', 'VotingWeighted']


In [ ]:
for name, model in ml_models_part1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: RandomForest
Accuracy      : 0.8802
Sensitivity   : 0.8802
Specificity   : 0.9900
G-Mean        : 0.9335
Type I Error  : 0.0100
Type II Error : 0.1198
F1 Score      : 0.8772
AUROC         : 0.9921

Model: ExtraTrees
Accuracy      : 0.8620
Sensitivity   : 0.8620
Specificity   : 0.9885
G-Mean        : 0.9231
Type I Error  : 0.0115
Type II Error : 0.1380
F1 Score      : 0.8600
AUROC         : 0.9885

Model: Bagging
Accuracy      : 0.8670
Sensitivity   : 0.8670
Specificity   : 0.9889
G-Mean        : 0.9260
Type I Error  : 0.0111
Type II Error : 0.1330
F1 Score      : 0.8634
AUROC         : 0.9838

Model: GradientBoosting
Accuracy      : 0.8024
Sensitivity   : 0.8024
Specificity   : 0.9835
G-Mean        : 0.8884
Type I Error  : 0.0165
Type II Error : 0.1976
F1 Score      : 0.7919
AUROC         : 0.9855


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Model: LogisticRegression
Accuracy      : 0.7156
Sensitivity   : 0.7156
Specificity   : 0.9763
G-Mean        : 0.8359
Type I Error  : 0.0237
Type II Error : 0.2844
F1 Score      : 0.7125
AUROC         : 0.9731

Model: RidgeClassifier
Accuracy      : 0.5372
Sensitivity   : 0.5372
Specificity   : 0.9614
G-Mean        : 0.7187
Type I Error  : 0.0386
Type II Error : 0.4628
F1 Score      : 0.5079
AUROC         : 0.0000

Model: DecisionTree
Accuracy      : 0.8402
Sensitivity   : 0.8402
Specificity   : 0.9867
G-Mean        : 0.9105
Type I Error  : 0.0133
Type II Error : 0.1598
F1 Score      : 0.8401
AUROC         : 0.9151

Model: NaiveBayes
Accuracy      : 0.4378
Sensitivity   : 0.4378
Specificity   : 0.9532
G-Mean        : 0.6460
Type I Error  : 0.0468
Type II Error : 0.5622
F1 Score      : 0.4325
AUROC         : 0.9171

Model: Perceptron
Accuracy      : 0.3649
Sensitivity   : 0.3649
Specificity   : 0.9471
G-Mean        : 0.5878
Type I Error  : 0.0529
Type II Error : 0.6351
F1 Score      : 

In [ ]:
ml_models_part2_sub1 = {}
ml_models_part2_sub2 = {}

model_names_part2 = list(ml_models_part2.keys())
num_models_part2 = len(model_names_part2)

half_point = num_models_part2 // 2

for i, name in enumerate(model_names_part2):
    if i < half_point:
        ml_models_part2_sub1[name] = ml_models_part2[name]
    else:
        ml_models_part2_sub2[name] = ml_models_part2[name]

print(f"ml_models_part2_sub1 contains {len(ml_models_part2_sub1)} models: {list(ml_models_part2_sub1.keys())}")
print(f"ml_models_part2_sub2 contains {len(ml_models_part2_sub2)} models: {list(ml_models_part2_sub2.keys())}")

ml_models_part2_sub1 contains 3 models: ['PassiveAggressive', 'LDA', 'QDA']
ml_models_part2_sub2 contains 4 models: ['LightGBM', 'VotingSoft', 'VotingHard', 'VotingWeighted']


In [ ]:
for name, model in ml_models_part2_sub1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: PassiveAggressive
Accuracy      : 0.3059
Sensitivity   : 0.3059
Specificity   : 0.9422
G-Mean        : 0.5368
Type I Error  : 0.0578
Type II Error : 0.6941
F1 Score      : 0.2375
AUROC         : 0.0000

Model: LDA
Accuracy      : 0.6007
Sensitivity   : 0.6006
Specificity   : 0.9667
G-Mean        : 0.7620
Type I Error  : 0.0333
Type II Error : 0.3994
F1 Score      : 0.5950
AUROC         : 0.9538

Model: QDA
Accuracy      : 0.5708
Sensitivity   : 0.5708
Specificity   : 0.9642
G-Mean        : 0.7419
Type I Error  : 0.0358
Type II Error : 0.4292
F1 Score      : 0.5657
AUROC         : 0.9501


In [ ]:
ml_models_part2_sub2_sub1 = {}
ml_models_part2_sub2_sub2 = {}

model_names_part2_sub2 = list(ml_models_part2_sub2.keys())
num_models_part2_sub2 = len(model_names_part2_sub2)

half_point_sub2 = num_models_part2_sub2 // 2

for i, name in enumerate(model_names_part2_sub2):
    if i < half_point_sub2:
        ml_models_part2_sub2_sub1[name] = ml_models_part2_sub2[name]
    else:
        ml_models_part2_sub2_sub2[name] = ml_models_part2_sub2[name]

print(f"ml_models_part2_sub2_sub1 contains {len(ml_models_part2_sub2_sub1)} models: {list(ml_models_part2_sub2_sub1.keys())}")
print(f"ml_models_part2_sub2_sub2 contains {len(ml_models_part2_sub2_sub2)} models: {list(ml_models_part2_sub2_sub2.keys())}")



ml_models_part2_sub2_sub1 contains 2 models: ['LightGBM', 'VotingSoft']
ml_models_part2_sub2_sub2 contains 2 models: ['VotingHard', 'VotingWeighted']


In [ ]:
for name, model in ml_models_part2_sub2_sub1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: LightGBM
Accuracy      : 0.8488
Sensitivity   : 0.8488
Specificity   : 0.9874
G-Mean        : 0.9155
Type I Error  : 0.0126
Type II Error : 0.1512
F1 Score      : 0.8433
AUROC         : 0.9909

Model: VotingSoft
Accuracy      : 0.8837
Sensitivity   : 0.8837
Specificity   : 0.9903
G-Mean        : 0.9355
Type I Error  : 0.0097
Type II Error : 0.1163
F1 Score      : 0.8801
AUROC         : 0.9932


In [ ]:
for name, model in ml_models_part2_sub2_sub2.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: VotingHard
Accuracy      : 0.8599
Sensitivity   : 0.8599
Specificity   : 0.9883
G-Mean        : 0.9219
Type I Error  : 0.0117
Type II Error : 0.1401
F1 Score      : 0.8518
AUROC         : 0.0000

Model: VotingWeighted
Accuracy      : 0.8722
Sensitivity   : 0.8722
Specificity   : 0.9894
G-Mean        : 0.9290
Type I Error  : 0.0106
Type II Error : 0.1278
F1 Score      : 0.8672
AUROC         : 0.9928


In [ ]:

stacking_1 = {
    "Stacking_LR": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        final_estimator=LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}

for name, model in stacking_1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: Stacking_LR
Accuracy      : 0.8889
Sensitivity   : 0.8889
Specificity   : 0.9907
G-Mean        : 0.9385
Type I Error  : 0.0093
Type II Error : 0.1111
F1 Score      : 0.8868
AUROC         : 0.9939


In [ ]:
stacking_2 = {
    "Stacking_LGBM": StackingClassifier(
        estimators=[
            ('lda', LinearDiscriminantAnalysis()),
            ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
            ('dt', DecisionTreeClassifier(random_state=42)), # Added DecisionTreeClassifier as a replacement
            ('nb', GaussianNB())
        ],
        final_estimator=LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
   )
}

for name, model in stacking_2.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model: Stacking_LGBM
Accuracy      : 0.8529
Sensitivity   : 0.8529
Specificity   : 0.9877
G-Mean        : 0.9178
Type I Error  : 0.0123
Type II Error : 0.1471
F1 Score      : 0.8488
AUROC         : 0.9905


In [ ]:
stacking_3 = {
    "Stacking_GB": StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42)),
            ('sgd', SGDClassifier(random_state=42, loss='modified_huber')),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_3.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_GB
Accuracy      : 0.8828
Sensitivity   : 0.8828
Specificity   : 0.9902
G-Mean        : 0.9350
Type I Error  : 0.0098
Type II Error : 0.1172
F1 Score      : 0.8805
AUROC         : 0.9934


In [ ]:
stacking_4 = {
    "Stacking_RF": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB()),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42))
        ],
        final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_4.items():
    run_model(name, model, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: Stacking_RF
Accuracy      : 0.8895
Sensitivity   : 0.8895
Specificity   : 0.9908
G-Mean        : 0.9388
Type I Error  : 0.0092
Type II Error : 0.1105
F1 Score      : 0.8869
AUROC         : 0.9931


In [ ]:
stacking_5 = {
    "Stacking_ET": StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=ExtraTreesClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_5.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_ET
Accuracy      : 0.8506
Sensitivity   : 0.8506
Specificity   : 0.9875
G-Mean        : 0.9165
Type I Error  : 0.0125
Type II Error : 0.1494
F1 Score      : 0.8455
AUROC         : 0.9908


**Outputs are collected manually**